### Hospitais e leitos — CNES

Download de Hospitais e Leitos (recorte do CNES), 2020 a 2024.
Notebook:
    %pip install pandas pyarrow
    %run baixar_cnes_leitos_2020_2024.py
Fonte: https://dadosabertos.saude.gov.br/dataset/hospitais-e-leitos
Nao e o cadastro completo de todos os estabelecimentos CNES.
Cada arquivo anual contem competencias mensais (COMP=AAAAMM).
Leitos cadastrados nao representam ocupacao. Nao some meses como estoque anual.
CSV original preservado; Parquet com nomes padronizados e valores como texto.
Nao remove linhas, nao deduplica, nao preenche meses ou converte ausentes em zero.
Para uma nova coleta, altere PASTA. Por padrao reutiliza Parquets validados.
Nao execute duas instancias simultaneamente na mesma pasta.

In [1]:
import csv
import hashlib
import json
import re
import time
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PASTA = BASE / 'dados_cnes_leitos'
ANOS = range(2020, 2025)
LOTE = 20000
URLS = {a: f'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/Leitos_SUS/Leitos_{a}.csv' for a in ANOS}

In [3]:
def salvar_json(p, dados):
    tmp = p.with_suffix(p.suffix + '.part')
    tmp.write_text(json.dumps(dados, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(p)


def sha256(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda: f.read(1024**2), b''): h.update(b)
    return h.hexdigest()


def nome_coluna(s):
    s = unicodedata.normalize('NFKD', s.strip()).encode('ascii', 'ignore').decode().upper()
    s = re.sub(r'[^A-Z0-9]+', '_', s).strip('_')
    return {'LEITOS_EXISTENTE': 'LEITOS_EXISTENTES'}.get(s, s)


def baixar(url, destino):
    parcial = destino.with_suffix('.csv.part')
    for tentativa in range(1, 4):
        try:
            req = Request(url, headers={'User-Agent': 'Observatorio-COVID-Brasil/1.0', 'Accept-Encoding': 'identity'})
            with urlopen(req, timeout=90) as r, parcial.open('wb') as f:
                if r.status != 200: raise IOError(f'HTTP {r.status}: esperado arquivo completo.')
                if 'html' in r.headers.get('Content-Type', '').lower(): raise ValueError('Servidor retornou HTML.')
                total = int(r.headers.get('Content-Length', 0))
                recebido = 0
                ultimo = time.monotonic()
                while True:
                    b = r.read(1024**2)
                    if not b: break
                    f.write(b); recebido += len(b)
                    if time.monotonic() - ultimo > 3:
                        print(f'  {recebido/1024**2:.1f} MB baixados', flush=True)
                        ultimo = time.monotonic()
                metadados = {'etag': r.headers.get('ETag'), 'last_modified': r.headers.get('Last-Modified')}
            if not recebido or (total and recebido != total): raise IOError('Download vazio/incompleto.')
            parcial.replace(destino)
            return metadados
        except Exception as e:
            print(f'  Tentativa {tentativa}/3: {e}', flush=True)
            if tentativa == 3: raise
            time.sleep(3 * tentativa)


def converter(origem, destino, ano):
    parcial = destino.with_suffix('.parquet.part')
    for encoding in ('utf-8-sig', 'latin-1'):
        writer = None
        try:
            total = invalidas = outro_ano = duplicadas = cnes_invalidos = 0
            competencias = Counter(); chaves = set(); lote = []
            with origem.open(encoding=encoding, newline='') as f:
                # CSV oficial usa virgula; csv.reader respeita aspas e quebras internas.
                leitor = csv.reader(f, delimiter=',', strict=True)
                originais = next(leitor)
                colunas = [nome_coluna(c) for c in originais]
                if not all(colunas) or len(set(colunas)) != len(colunas): raise ValueError('Cabecalho vazio ou ambiguo.')
                obrigatorias = {'COMP', 'CNES', 'UF', 'MUNICIPIO', 'LEITOS_EXISTENTES', 'LEITOS_SUS', 'UTI_TOTAL_EXIST', 'UTI_TOTAL_SUS'}
                if not obrigatorias.issubset(colunas): raise ValueError('Layout inesperado: faltam ' + str(sorted(obrigatorias-set(colunas))))
                esquema = pa.schema([(c, pa.string()) for c in colunas])
                writer = pq.ParquetWriter(parcial, esquema, compression='snappy')
                for linha in leitor:
                    if not linha: continue  # Linha fisicamente vazia, nao um registro.
                    if len(linha) != len(colunas): raise ValueError(f'Linha CSV {leitor.line_num}: {len(linha)} campos; esperado {len(colunas)}.')
                    registro = dict(zip(colunas, linha))
                    comp, cnes = registro['COMP'].strip(), registro['CNES'].strip()
                    valido = bool(re.fullmatch(r'\d{4}(0[1-9]|1[0-2])', comp))
                    invalidas += int(not valido)
                    outro_ano += int(valido and comp[:4] != str(ano))
                    cnes_invalidos += int(not re.fullmatch(r'\d{7}', cnes))
                    competencias[comp] += 1
                    chave = (comp, cnes)
                    duplicadas += int(chave in chaves)
                    chaves.add(chave)
                    lote.append(registro); total += 1
                    if len(lote) >= LOTE:
                        writer.write_table(pa.Table.from_pylist(lote, schema=esquema)); lote.clear()
                        print(f'  {ano}: {total:,} registros convertidos', flush=True)
                if lote: writer.write_table(pa.Table.from_pylist(lote, schema=esquema))
            writer.close(); writer = None
            if not total: raise ValueError('CSV sem registros.')
            if pq.read_metadata(parcial).num_rows != total: raise ValueError('Contagem Parquet divergente.')
            parcial.replace(destino)
            return {'registros': total, 'colunas': len(colunas), 'encoding': encoding,
                    'mapeamento_colunas': dict(zip(originais, colunas)),
                    'registros_por_competencia': dict(sorted(competencias.items())),
                    'competencias_ausentes': sorted({f'{ano}{m:02}' for m in range(1,13)}-set(competencias)),
                    'competencias_invalidas': invalidas, 'registros_de_outro_ano': outro_ano,
                    'cnes_fora_padrao_7_digitos': cnes_invalidos,
                    'repeticoes_cnes_comp_excedentes': duplicadas}
        except UnicodeDecodeError:
            if encoding == 'latin-1': raise
            print('  Reiniciando leitura com Latin-1...', flush=True)
        finally:
            if writer is not None: writer.close()


def main():
    for sub in ('brutos', 'parquet'): (PASTA/sub).mkdir(parents=True, exist_ok=True)
    manifesto = PASTA/'manifesto_download.json'
    estados = json.loads(manifesto.read_text(encoding='utf-8')) if manifesto.exists() else {}
    falhas = []
    for ano in ANOS:
        url = URLS[ano]; csv_path = PASTA/'brutos'/f'leitos_{ano}.csv'
        destino = PASTA/'parquet'/f'cnes_leitos_{ano}.parquet'
        anterior = estados.get(str(ano), {})
        try:
            if anterior.get('status') == 'concluido' and anterior.get('url') == url and destino.exists():
                if sha256(destino) == anterior.get('sha256_parquet') and pq.read_metadata(destino).num_rows == anterior.get('registros'):
                    print(f'{ano}: Parquet concluido e validado.', flush=True); continue
            print(f'\n{ano}: baixando CSV oficial...', flush=True)
            meta = baixar(url, csv_path)
            info = converter(csv_path, destino, ano)
            estados[str(ano)] = {'status': 'concluido', 'url': url,
                'coletado_utc': datetime.now(timezone.utc).isoformat(),
                'sha256_csv': sha256(csv_path), 'sha256_parquet': sha256(destino), **meta, **info}
            if info['competencias_ausentes'] or info['competencias_invalidas'] or info['registros_de_outro_ano'] or info['repeticoes_cnes_comp_excedentes']:
                print('  Confira os avisos de competencias/duplicidades no manifesto.', flush=True)
        except Exception as e:
            falhas.append(ano)
            estados[str(ano)] = {'status': 'erro', 'url': url, 'erro': str(e)}
            print(f'Falha em {ano}: {e}', flush=True)
        salvar_json(manifesto, estados)
    resumo = [{k: v for k,v in {'ano':a, **estados.get(str(a), {})}.items() if not isinstance(v, (dict,list))} for a in ANOS]
    pd.DataFrame(resumo).to_csv(PASTA/'resumo_download.csv', index=False, sep=';', encoding='utf-8-sig')
    if falhas: raise RuntimeError(f'Coleta incompleta: {falhas}. Consulte {manifesto}. Execute novamente para tentar esses anos.')
    print('\nConcluido. Parquets em: ' + str(PASTA/'parquet'))
    print('COMP identifica o mes. Analise leitos por competencia; nao some estoques de meses distintos.')
    return estados


if __name__ == '__main__':
    resultado_cnes = main()



2020: baixando CSV oficial...
  2020: 20,000 registros convertidos
  2020: 40,000 registros convertidos
  2020: 60,000 registros convertidos
  2020: 80,000 registros convertidos

2021: baixando CSV oficial...
  2021: 20,000 registros convertidos
  2021: 40,000 registros convertidos
  2021: 60,000 registros convertidos
  2021: 80,000 registros convertidos

2022: baixando CSV oficial...
  2022: 20,000 registros convertidos
  2022: 40,000 registros convertidos
  2022: 60,000 registros convertidos
  2022: 80,000 registros convertidos

2023: baixando CSV oficial...
  Reiniciando leitura com Latin-1...
  2023: 20,000 registros convertidos
  2023: 40,000 registros convertidos
  2023: 60,000 registros convertidos
  2023: 80,000 registros convertidos

2024: baixando CSV oficial...
  Reiniciando leitura com Latin-1...
  2024: 20,000 registros convertidos
  2024: 40,000 registros convertidos
  2024: 60,000 registros convertidos
  2024: 80,000 registros convertidos

Concluido. Parquets em: d:\Cle